# 제품 이상여부 판별 프로젝트


## 1. 데이터 불러오기


### 필수 라이브러리


In [126]:
import os
from pprint import pprint

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV

from tqdm import tqdm

In [127]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)

### 데이터 읽어오기


In [128]:
ROOT_DIR = "data"
RANDOM_STATE = 110

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
list(train_data)
print("The total number of colums: " + str(len(train_data.columns)))

The total number of colums: 464


## 1st Data Preprocessing

#### Identify those columns with NA included

In [129]:
na_cal = []

for i in train_data.columns:
    if (train_data[i].isna().sum()) > 0:
        na_cal.append(i)

print("Total number of columns with NA: " + str(len(na_cal)))

train_data[na_cal] = train_data[na_cal].fillna(0)

Total number of columns with NA: 286


#### Identify those columns with one value duplicated to every row

In [130]:
one_val_duplicated = []
for i in train_data.columns:
    if (train_data[i].nunique()) == 1:
        one_val_duplicated.append(i)
        
print("Total number of columns with only one val: " + str(len(one_val_duplicated)))

Total number of columns with only one val: 313


#### Identify those columns with unique values for every row

In [131]:
# "Number of unique entries = Num rows" ==> "Unique value for every row"
num_rows = len(train_data)
unique_every_row = []
for i in train_data.columns:
    if (train_data[i].value_counts().size == num_rows):
        unique_every_row.append(i)
        
print("Total number of columns with unique values for every row: " + str(len(unique_every_row)))

Total number of columns with unique values for every row: 0


In [132]:
multiple_types = []
for i in train_data.columns:
    if (len(set(train_data[i].apply(type))) > 1):
        multiple_types.append(i)

print(multiple_types)
print("Total number of columns with multiple datatypes: " + str(len(multiple_types)))

['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam', 'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave', 'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2']
Total number of columns with multiple datatypes: 8


In [133]:
# Columns with "OK" and numbers are mixed --> to be processed ("OK" to NaN):
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2

# Columns with "OK" and NaN are mixed:
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam
#     GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave
#     GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2

#### Automating to replace of "OK"s with null/NaN as instructed

In [134]:
ok_2_nan = ["HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam", 
            "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1", 
            "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2"]

for i in ok_2_nan:
    train_data[i] = train_data[i].replace('OK', np.nan)
    train_data[i] = train_data[i].astype(float)

#### ACTION!!!!

In [135]:
train_data.drop(columns = one_val_duplicated, inplace = True)
train_data.drop(columns = unique_every_row, inplace = True)

print("Number of columns: " + str(len(train_data.columns)))

Number of columns: 151


### 언더 샘플링


데이타 불균형을 해결하기 위해 언더 샘플링을 진행합니다.


In [136]:
normal_ratio = 1.0  # 1.0 means 1:1 ratio

df_normal = train_data[train_data["target"] == "Normal"]
df_abnormal = train_data[train_data["target"] == "AbNormal"]

num_normal = len(df_normal)
num_abnormal = len(df_abnormal)
print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}")

df_normal = df_normal.sample(n=int(num_abnormal * normal_ratio), replace=False, random_state=RANDOM_STATE)
df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
df_concat.value_counts("target")

  Total: Normal: 38156, AbNormal: 2350


target
AbNormal    2350
Normal      2350
Name: count, dtype: int64

### 데이터 분할


In [137]:
df_train, df_val = train_test_split(
    df_concat,
    test_size=0.3,
    stratify=df_concat["target"],
    random_state=RANDOM_STATE,
)


def print_stats(df: pd.DataFrame):
    num_normal = len(df[df["target"] == "Normal"])
    num_abnormal = len(df[df["target"] == "AbNormal"])

    print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}" + f" ratio: {num_abnormal/num_normal}")


# Print statistics
print(f"  \tAbnormal\tNormal")
print_stats(df_train)
print_stats(df_val)

  	Abnormal	Normal
  Total: Normal: 1645, AbNormal: 1645 ratio: 1.0
  Total: Normal: 705, AbNormal: 705 ratio: 1.0


## 3. 모델 학습


### 모델 학습


#### Hyperparameter Tuning

In [138]:
features = []

for col in df_train.columns:
    try:
        df_train[col] = df_train[col].astype(int)
        features.append(col)
    except:
        continue

train_x = df_train[features]
train_y = df_train["target"]

params = { 'n_estimators' : [10, 30, 50, 80, 100, 150],
           'max_depth' : [6, 8, 10, 12, 15, 20],
           'min_samples_leaf' : [8, 12, 18, 20],
           'min_samples_split' : [8, 16, 20, 25],
           'n_jobs' : [-1],
           'random_state' : [RANDOM_STATE],
            }

# RandomForestClassifier 객체 생성 후 GridSearchCV 수행
rf_clf = RandomForestClassifier(random_state = 0, n_jobs = -1)
grid_cv = GridSearchCV(rf_clf, param_grid = params, cv = 3, n_jobs = -1)
grid_cv.fit(train_x, train_y)

print('최적 하이퍼 파라미터: ', grid_cv.best_params_)
print('최고 예측 정확도: {:.4f}'.format(grid_cv.best_score_))


최적 하이퍼 파라미터:  {'max_depth': 15, 'min_samples_leaf': 8, 'min_samples_split': 20, 'n_estimators': 30, 'n_jobs': -1, 'random_state': 110}
최고 예측 정확도: 0.6164


In [139]:
pre_calibration_model = RandomForestClassifier(n_estimators = 30, 
                                               max_depth = 15, 
                                               min_samples_leaf = 8,
                                               min_samples_split = 20,
                                               n_jobs = -1,
                                               random_state = RANDOM_STATE).fit(train_x, train_y)

In [140]:
model = CalibratedClassifierCV(pre_calibration_model, method='sigmoid', cv=5)
model.fit(train_x, train_y)

CalibratedClassifierCV(cv=5,
                       estimator=RandomForestClassifier(max_depth=15,
                                                        min_samples_leaf=8,
                                                        min_samples_split=20,
                                                        n_estimators=30,
                                                        n_jobs=-1,
                                                        random_state=110))

## 4. 제출하기


### 테스트 데이터 예측


테스트 데이터 불러오기


In [141]:
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

In [142]:
df_test_x = test_data[features]

for col in df_test_x.columns:
    try:
        df_test_x.loc[:, col] = df_test_x[col].astype(int)
    except:
        continue

In [143]:
test_pred = model.predict(df_test_x)
test_pred

array(['AbNormal', 'Normal', 'AbNormal', ..., 'Normal', 'AbNormal',
       'Normal'], dtype=object)

### 제출 파일 작성


In [144]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
df_sub = pd.read_csv("submission.csv")
df_sub["target"] = test_pred

# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)

**우측 상단의 제출 버튼을 클릭해 결과를 확인하세요**
